# 5.2 K-Means Clustering

Notebook ini menjalankan **K-Means Clustering** pada dua skenario:

| Skenario | Input | Keterangan |
|----------|-------|------------|
| **Skenario 1** | 68 fitur asli | Tanpa reduksi dimensi |
| **Skenario 2** | 37 fitur PCA | Dengan reduksi dimensi |

Untuk masing-masing skenario, dicari **jumlah cluster optimal (K)** menggunakan **Elbow Method** dan **Silhouette Score**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

## 5.2.1 Memuat Data

Data 68 fitur asli + data 37 fitur PCA.

In [ ]:
# === SKENARIO 1: 68 fitur asli ===
POLLUTANTS = ['CO', 'NO2', 'SO2']
feature_dfs = []

for p in POLLUTANTS:
    path = f'{p}_Jabon_TSFEL.csv'
    try:
        df = pd.read_csv(path)
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        df_features = df[numeric_cols].add_suffix(f'_{p}')
        feature_dfs.append(df_features)
    except FileNotFoundError:
        print(f'PERINGATAN: {path} tidak ditemukan, menggunakan data dummy')
        np.random.seed(42)
        df_features = pd.DataFrame(
            np.random.randn(1, 68),
            columns=[f'fitur{i+1}_{p}' for i in range(68)]
        )
        feature_dfs.append(df_features)

X_68 = pd.concat(feature_dfs, axis=1)
scaler_68 = StandardScaler()
X_68_scaled = scaler_68.fit_transform(X_68)
print(f'Skenario 1: {X_68_scaled.shape[1]} fitur')

In [ ]:
# === SKENARIO 2: 37 fitur PCA ===
try:
    X_pca = pd.read_csv('jabon_pca_37fitur.csv').values
    print(f'Skenario 2: {X_pca.shape[1]} fitur PCA')
except FileNotFoundError:
    print('File PCA tidak ditemukan. Jalankan notebook 5.1 terlebih dahulu!')
    print('Menggunakan data dummy untuk demonstrasi.')
    np.random.seed(42)
    X_pca = np.random.randn(1, 37)
    print(f'Skenario 2: {X_pca.shape[1]} fitur PCA (dummy)')

## 5.2.2 Menentukan K Optimal

### Elbow Method
Mencari *elbow* pada kurva inertia (Within-Cluster Sum of Squares). Titik di mana penurunan inertia mulai melambat menandakan K optimal.

### Silhouette Score
Mengukur seberapa mirip sebuah titik dengan cluster-nya sendiri dibandingkan cluster lain. Skor mendekati 1 = cluster yang baik.

In [ ]:
K_range = range(2, 11)

def find_optimal_k(X, label_prefix=''):
    """Jalankan K-Means untuk K=2..10, kembalikan inertia + silhouette."""
    inertias = []
    silhouettes = []
    
    for k in K_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
        labels = km.fit_predict(X)
        inertias.append(km.inertia_)
        sil = silhouette_score(X, labels)
        silhouettes.append(sil)
        print(f'  {label_prefix} K={k}: inertia={km.inertia_:.2f}, silhouette={sil:.4f}')
    
    best_k = list(K_range)[np.argmax(silhouettes)]
    best_sil = max(silhouettes)
    return inertias, silhouettes, best_k, best_sil

In [ ]:
print('=== SKENARIO 1: 68 Fitur ===')
inertias_68, sil_68, best_k_68, best_sil_68 = find_optimal_k(X_68_scaled, '68fitur')
print(f'\n>>> K terbaik (68 fitur): K={best_k_68}, silhouette={best_sil_68:.4f}\n')

print('=== SKENARIO 2: 37 Fitur PCA ===')
inertias_pca, sil_pca, best_k_pca, best_sil_pca = find_optimal_k(X_pca, '37PCA')
print(f'\n>>> K terbaik (37 PCA): K={best_k_pca}, silhouette={best_sil_pca:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Elbow
ax1.plot(list(K_range), inertias_68, 'bo-', label='68 Fitur', linewidth=2)
ax1.plot(list(K_range), inertias_pca, 'rs--', label='37 PCA', linewidth=2)
ax1.set_xlabel('Jumlah Cluster (K)')
ax1.set_ylabel('Inertia (WCSS)')
ax1.set_title('Elbow Method')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Silhouette
ax2.plot(list(K_range), sil_68, 'bo-', label='68 Fitur', linewidth=2)
ax2.plot(list(K_range), sil_pca, 'rs--', label='37 PCA', linewidth=2)
ax2.set_xlabel('Jumlah Cluster (K)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5.2.3 Clustering dengan K Optimal

Jalankan K-Means dengan K terbaik dari masing-masing skenario.

In [ ]:
# Skenario 1: 68 fitur, K terbaik
km_68 = KMeans(n_clusters=best_k_68, random_state=42, n_init=10)
labels_68 = km_68.fit_predict(X_68_scaled)
print(f'Skenario 1 (68 fitur, K={best_k_68}):')
for c in range(best_k_68):
    print(f'  Cluster {c}: {np.sum(labels_68 == c)} data')

# Skenario 2: 37 PCA, K terbaik
km_pca = KMeans(n_clusters=best_k_pca, random_state=42, n_init=10)
labels_pca = km_pca.fit_predict(X_pca)
print(f'\nSkenario 2 (37 PCA, K={best_k_pca}):')
for c in range(best_k_pca):
    print(f'  Cluster {c}: {np.sum(labels_pca == c)} data')

## 5.2.4 Visualisasi Hasil Cluster (PCA 2D)

In [ ]:
from sklearn.decomposition import PCA

# Proyeksikan ke 2D untuk visualisasi
pca_2d = PCA(n_components=2)
X_68_2d = pca_2d.fit_transform(X_68_scaled)
X_pca_2d = pca_2d.fit_transform(X_pca)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Skenario 1
scatter1 = ax1.scatter(X_68_2d[:, 0], X_68_2d[:, 1], c=labels_68, cmap='viridis', s=50, alpha=0.7)
ax1.set_title(f'Skenario 1: 68 Fitur (K={best_k_68})')
ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')
plt.colorbar(scatter1, ax=ax1, label='Cluster')

# Skenario 2
scatter2 = ax2.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=labels_pca, cmap='viridis', s=50, alpha=0.7)
ax2.set_title(f'Skenario 2: 37 PCA (K={best_k_pca})')
ax2.set_xlabel('PC1')
ax2.set_ylabel('PC2')
plt.colorbar(scatter2, ax=ax2, label='Cluster')

plt.tight_layout()
plt.show()

## 5.2.5 Ringkasan Metrik Kedua Skenario

| Metrik | Skenario 1 (68 Fitur) | Skenario 2 (37 PCA) |
|--------|----------------------|--------------------|
| Jumlah fitur | 68 | 37 |
| K terbaik | - | - |
| Silhouette Score | - | - |
| Variance Explained | 100% | - |

In [ ]:
# Tabel komparasi
komparasi = pd.DataFrame({
    'Skenario': ['68 Fitur (Tanpa PCA)', '37 Fitur (Dengan PCA)'],
    'Jumlah Fitur': [68, 37],
    'K Terbaik': [best_k_68, best_k_pca],
    'Silhouette Score': [f'{best_sil_68:.4f}', f'{best_sil_pca:.4f}'],
})
komparasi

## 5.2.6 Kesimpulan

1. Silhouette Score yang lebih tinggi menunjukkan cluster yang lebih terpisah dan compact.
2. Jika Skenario 2 (PCA) menghasilkan silhouette yang lebih baik atau setara → PCA efektif mereduksi noise.
3. Jika Skenario 1 lebih baik → fitur asli masih lebih informatif.
4. Komparasi lengkap ada di notebook 5.3.